In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn import feature_extraction, model_selection, metrics, svm
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline


In [ ]:
data = pd.read_excel('/content/sample_data/Spam Email Detection.xlsx')

In [ ]:
count_Class = pd.value_counts(data["v1"], sort=True)

# Bar chart
count_Class.plot(kind='bar', color=["red", "green"])
plt.title('Bar chart')
plt.show()

# Pie chart
count_Class.plot(kind='pie', autopct='%1.0f%%')
plt.title('Pie chart')
plt.ylabel('')
plt.show()


In [ ]:
count_Class = pd.value_counts(data["v1"], sort=True)

# Bar chart
count_Class.plot(kind='bar', color=["red", "green"])
plt.title('Bar chart')
plt.show()

# Pie chart
count_Class.plot(kind='pie', autopct='%1.0f%%')
plt.title('Pie chart')
plt.ylabel('')
plt.show()


In [ ]:
data['v2'] = data['v2'].astype(str)

count1 = Counter(" ".join(data[data['v1']=='ham']["v2"]).split()).most_common(20)
df1 = pd.DataFrame.from_dict(count1)
df1 = df1.rename(columns={0: "words in non-spam", 1: "count"})

count2 = Counter(" ".join(data[data['v1']=='spam']["v2"]).split()).most_common(20)
df2 = pd.DataFrame.from_dict(count2)
df2 = df2.rename(columns={0: "words in spam", 1: "count_"})

In [ ]:
df1.plot.bar(legend=False)
y_pos = np.arange(len(df1["words in non-spam"]))
plt.xticks(y_pos, df1["words in non-spam"])
plt.title('More frequent words in non-spam messages')
plt.xlabel('words')
plt.ylabel('number')
plt.show()


In [ ]:
df2.plot.bar(legend=False, color='orange')
y_pos = np.arange(len(df2["words in spam"]))
plt.xticks(y_pos, df2["words in spam"])
plt.title('More frequent words in spam messages')
plt.xlabel('words')
plt.ylabel('number')
plt.show()


In [ ]:
f = feature_extraction.text.CountVectorizer(stop_words='english')
X = f.fit_transform(data["v2"])

data["v1"] = data["v1"].map({'spam': 1, 'ham': 0})

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, data['v1'], test_size=0.33, random_state=42
)


In [ ]:
list_C = np.arange(50, 1000, 50)
score_train = np.zeros(len(list_C))
score_test = np.zeros(len(list_C))
recall_test = np.zeros(len(list_C))
precision_test = np.zeros(len(list_C))

count = 0

for C in list_C:
    svc = svm.SVC(C=C)
    svc.fit(X_train, y_train)

    score_train[count] = svc.score(X_train, y_train)
    score_test[count] = svc.score(X_test, y_test)
    recall_test[count] = metrics.recall_score(y_test, svc.predict(X_test))
    precision_test[count] = metrics.precision_score(y_test, svc.predict(X_test))

    count += 1


In [ ]:
matrix = np.matrix(np.c_[list_C, score_train, score_test, recall_test, precision_test])

models = pd.DataFrame(
    data=matrix,
    columns=['C', 'Train Accuracy', 'Test Accuracy', 'Test Recall', 'Test Precision']
)

# Find the maximum precision achieved
max_precision = models['Test Precision'].max()
# Filter for models that achieved this maximum precision
best_models_by_precision = models[models['Test Precision'] == max_precision]
# Among those, pick the one with the highest Test Accuracy
best_index = best_models_by_precision['Test Accuracy'].idxmax()

svc = svm.SVC(C=list_C[best_index])
svc.fit(X_train, y_train)

print("Best Model Parameters:")
print(models.iloc[best_index, :])

In [ ]:
m_confusion_test = metrics.confusion_matrix(y_test, svc.predict(X_test))

confusion_df = pd.DataFrame(
    data=m_confusion_test,
    columns=['Predicted Non-Spam', 'Predicted Spam'],
    index=['Actual Non-Spam', 'Actual Spam']
)

print("\nConfusion Matrix:")
print(confusion_df)


In [ ]:
mytest = "you have won a lottery of $2000. to claim it reply to this email"
Y = [mytest]

f_final = feature_extraction.text.CountVectorizer(stop_words='english')
f_final.fit(data["v2"])
X_input = f_final.transform(Y)

res = svc.predict(X_input)

if res == 0:
    print("\nResult: This is a non-spam email")
else:
    print("\nResult: This is a spam email")
